In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, DistanceMetric

pd.set_option("display.max_columns", 25)

In [ ]:
df = pd.read_csv("data/raw/patients.csv", index_col="index")

In [ ]:
df.head(3)

# 1. Data Cleaning

In [ ]:
df.info()

In [ ]:
df.describe() # Features - No Outliers / Impossible Values Appear

In [ ]:
df["Level"].describe() # Target Variable

Checking for Missing Values

In [ ]:
df.isnull().sum()

Checking for Duplicates

In [ ]:
df.duplicated().sum()

Checking for Format Issues
(Genders: Only 2 should exist)

In [ ]:
df["Gender"].unique()

# 2. Balance
Target balance as rougly equal height of all classes

In [ ]:
plt.figure(figsize=(4,3))

sns.countplot(
    data=df,
    x="Level"
)
plt.title("Level Distribution", fontsize=12, fontweight="bold")

# 3. EDA

## 3.1 Histograms / Bar Graph
Analyze Shape (Bell? Skewed?), Modality, Center, Spread

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
numeric_cols = [col for col in numeric_cols if col != "Gender"]#[:10]

fig, axes = plt.subplots(5, 5, figsize=(20,15))
axes = axes.flatten()

for i, feature in enumerate(numeric_cols):
    sns.histplot(
        data=df,
        x=feature,
        bins=df[feature].max() - df[feature].min(),
        kde=True,
        ax=axes[i]
    )

    axes[i].set_title(f"{feature} Distribution", fontsize=12, fontweight="bold")

    axes[i].axvline(
        df[feature].mean(),
        color='red',
        linestyle='--',
        label=f"Mean: {df[feature].mean():.1f}"
    )

    axes[i].legend()

sns.countplot(
    data=df,
    x="Gender",
    ax=axes[22]
)
axes[22].set_title("Gender Distribution", fontsize=12, fontweight="bold")
axes[22].set_xticks([0,1], ["Male", "Female"])

plt.tight_layout() # to prevent leaking of titles/axis labels
plt.show()

## 3.2 Box Plots
Analyze Outliers, Compare Distributions

Single Outlier in Age

In [ ]:
fig, axes = plt.subplots(5,5, figsize=(20,10))
axes = axes.flatten()

for i, feature in enumerate(numeric_cols):
    sns.boxplot(
        data=df,
        x=feature,
        ax=axes[i]
    )

    axes[i].set_title(f"{feature} Distribution")

    label = "Rating"
    if feature=="Age":
        label="Age"
    axes[i].set_xlabel(f"{label}")

plt.tight_layout()
plt.show()

# 4. Feature Selection

## 4.1 Visually - Box Plot (Feature Target)
The less overlap for each feature category the more related the feature to target
Alternative to Scatter Plot (require both feature and target to be continuous)

High Relation: Alcohol Use, Dust Allergy, Occupational Hazards, Obesity, Smoking. Fatigue

In [ ]:
fig, axes = plt.subplots(5,5, figsize=(20,10))
axes = axes.flatten()

for i, feature in enumerate(numeric_cols):
    sns.boxplot(
        data=df,
        x="Level",
        y=feature,
        ax=axes[i]
    )

    axes[i].set_title(f"{feature} Distribution")

    label = "Rating"
    if feature=="Age":
        label="Age"
    axes[i].set_xlabel(f"{label}")

plt.tight_layout()
plt.show()

## 4.2 Pearson Corelation Coefficient

In [ ]:
# Mapping Label Classes to Numeric Values as PCC works on Continuous variables 
df_encoded = df.copy()
df_encoded["Level"].replace({"Low":0, "Medium":1, "High":2}, inplace=True)

Selecting Features (threshold 50%)

In [ ]:
correlation_with_target = df_encoded[numeric_cols + ['Level']].corr(method='pearson')['Level'].round(4) * 100
print(correlation_with_target[ correlation_with_target>50 ].sort_values(ascending=False))

# 5. Feature Scaling
Required since KNN relies on distance for similarity <br>
Features with larger ranges will dominate otherwise <br>
Applied Standard Scaling (Mean=0 , Std=1)

In [ ]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# 6. Train/Test Split
- Train (80%) and Test (20%) <br>
- Train (70%) and Validation (30%) <br>
- Validation Set used to finetune hyperparameters (parameters of the learnign algorithm rather than of the mode like training iterations, learning rate)

In [ ]:
features = [col for col in correlation_with_target.index if col != "Level" ]
x_main, x_test, y_main, y_test = train_test_split(df[features], df["Level"], train_size=0.8, test_size=0.2, random_state=0)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split( x_main, y_main, train_size=0.7, test_size=0.3, random_state=0 )

# 7. Model Training

In [ ]:
knn = KNeighborsClassifier()
knn.fit(x_train, y_train)

In [ ]:
y_pred = knn.predict(x_test)

# 8. Model Evaluation
For Classification: Confusion Matrix + f1-score

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,6))
sns.heatmap(data=cm)
plt.title("Confusion Matirx", fontsize=15, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

# Misc: Testing Model with Different Parameters

In [ ]:
exp = {"minkowski":[], "cosine":[], "euclidean":[], "manhattan":[]}
for k in range(1, 11):
    for distmetric in exp.keys():
        knn = KNeighborsClassifier(n_neighbors=k, metric=distmetric)
        exp[distmetric].append( accuracy_score(y_test, knn.fit(x_train, y_train).predict(x_test)) )

In [ ]:
# Final Table - As we use more Neighbors, Performance Degrades
pd.DataFrame(exp, index=range(1,11))